# Processando Imagens com PIL (Pillow)

Antes de mergulhar na visão computacional, temos de cumprir alguns pré-requisitos. Um deles é ser capaz de manipular imagens utilizando PIL (Pillow).

## Introdução

A primeira coisa que temos de explorar é a leitura de imagens. No exemplo abaixo, definimos uma constante IMAGE_PATH que apenas aponta para uma imagem que estamos utilizando. Para de fato carregar a imagem, importamos a classe Image da biblioteca PIL. A partir dessa classe, chamamos o método open, passando como parâmetro o caminho até a imagem.

NOTA: O diretório raíz do projeto foi importado de um arquivo config.py dentro da pasta src.

Podemos acessar metadados da imagem importada. No exemplo abaixo, utilizamos a propriedade format para descobrir o formato da imagem (PNG). A propriedade mode informa o modo de cores da imagem. A imagem do exemplo está no modo RGBA. Além delas, temos a propriedade size que guarda uma tupla, na qual o primeiro item informa a largura da imagem em pixels e o segundo informa a altura da imagem em pixels.

In [2]:
from config import ROOT_DIR
from PIL import Image

IMAGE_PATH = ROOT_DIR / "data" / "folha-cafe.png"

image = Image.open(IMAGE_PATH)

print(f"Formato: {image.format}")
print(f"Modo: {image.mode}")
print(f"Tamanho (W, H): {image.size}")

Formato: PNG
Modo: RGBA
Tamanho (W, H): (537, 716)


### Conversão de Modos de Imagem

Ao trabalhar com redes neurais, devemos sempre utilizar o modo RGB. Já que a maioria das redes neurais convolucionais são projetadas para receber três canais, e não quatro. Podemos converter a imagem para outros modos utilizando o método convert do objeto da imagem, que recebe como primeiro parâmetro o modo que desejamos. No exemplo abaixo, convertemos a imagem para o modo RGB que tem três canais (R, G e B).

In [3]:
image = image.convert("RGB")

print(f"Modo: {image.mode}")

Modo: RGB


### Exibição de Imagem

Para visualizar a imagem, você pode chamar o método show do objeto da imagem, que abrirá um visualizador.

In [4]:
image.show()

### Salvando Imagens

Podemos salvar imagens utilizando o método save do objeto de imagem. O método recebe como primeiro parâmetro o caminho onde o arquivo será salvo.

In [5]:
FIGURES_DIR = ROOT_DIR / "assets" / "images"

image.save(FIGURES_DIR / "figura_01.png")

![Figura 1](assets/images/figura_01.png)

Ao salvar uma imagem, não é preciso salvar no mesmo formato que ela tem originalmente. Podemos por exemplo salvar em formato JPG, JPEG, WEBP, etc. Esses três formatos especificamente reagem a um parâmetro quality que deve ser um número de 1 até 95. O parâmetro quality define o nível de qualidade da imagem salva. Sendo 1 a qualidade mínima e 95 a qualidade máxima.

In [6]:
image.save(FIGURES_DIR / "figura_01.jpg", quality=10)

![Figura 5](assets/images/figura_01.jpg)

## Redimensionamento

Outra funcionalidade muito importante é copiar redimensionar as imagens. Para copiar, podemos chamar o método copy da imagem, que retorna um outro objeto de imagem exatamente igual. Isso pode ser essencial se quisermos manter a imagem original intacta.

Para redimensionar, podemos utilizar o método resize. O primeiro parâmetro deve ser uma tupla com dois itens, na qual o primeiro item é a largura desejada em pixels e o segundo item é a altura desejada em pixels. Também podemos escolher um método de redimensionamento que será passado como segundo parâmetro. No exemplo abaixo, utilizamos o método de redimensionamento de Lanczos.

In [7]:
image_resize = image.copy()
image_resize = image.resize((224, 224), Image.Resampling.LANCZOS)

image_resize.save(FIGURES_DIR / "figura_02.png")

![Figura 2](assets/images/figura_02.png)

Outro método que podemos utilizar é o método thumbnail que modifica a imagem in-place, então o objeto original será modificado. Diferente do resize que temos de reatribuir o valor da variável. O método resize redimensiona a imagem para conter o exato mesmo tamanho que especificamos ao chamá-lo, podendo distorcer a imagem. Já o método thumbnail redimensiona a imagem para conter no máximo os tamanhos especificados, preservando a integridade da imagem, sem distorção.

Como primeito parâmetro, o método thumbnail receberá uma tupla na qual o primeiro item é a largura máxima desejada e o segundo item é a altura máxima desejada. Nesse exemplo, a imagem não é quadrada. Então se a altura for maior que a largura, a altura será de 300 pixels e a largura será menor que 300 pixels. Nesse caso, a largura é de 225 pixels.

In [8]:
image_thumb = image.copy()
image_thumb.thumbnail((300, 300))

print(image_thumb.size)

image_thumb.save(FIGURES_DIR / "figura_03.png")


(225, 300)


![Figura 3](assets/images/figura_03.png)

## Corte de Imagem

Com o PIL, podemos cortar imagens utilizando o método crop dos objetos de imagem. No exemplo abaixo, fizemos algo interessante. Definimos tamanhos máximos para redimensionar em thumbnail, gerando novamente uma imagem como a do exemplo anterior (225, 300). Depois disso, calculamos quatro variáveis que representam os limites do corte. A variável x1 indica o início do corte horizontalmente, a variável x2 indica o fim do corte horizontalmente. A variável y1 indica o início do corte verticalmente e a variável y2 indica o fim do corte verticalmente. Fizemos um cálculo que resultará em uma caixa de corte que seleciona um quadrado perfeito exatamente no centro da imagem, resultando em uma imagem que deverá ser do tamanho (225, 225).

O método crop recebe como primeiro parâmetro uma tupla que contém exatamente esses limites de corte na ordem: Início horizontal, início vertical, fim horizontal, fim vertical. Portanto, na demonstração, definimos uma variável crop_box que ordena os limites calculados como o PIL pede e chamamos o método crop passando a tupla como parâmetro.

In [9]:
target_size = (300, 300)

image_thumb = image.copy()
image_thumb.thumbnail(target_size)

image_size = image_thumb.size
lower_size = min(image_size)

x1 = (image_size[0] - lower_size) / 2
x2 = x1 + lower_size
y1 = (image_size[1] - lower_size) / 2
y2 = y1 + lower_size

crop_box = (x1, y1, x2, y2)

image_crop = image_thumb.crop(crop_box)

print(target_size)
print(image_size)
print(crop_box)

image_crop.save(FIGURES_DIR / "figura_04.png")


(300, 300)
(225, 300)
(0.0, 37.5, 225.0, 262.5)


![Figura 4](assets/images/figura_04.png)

Todo o cálculo das coordenadas para a caixa é irrelevante para aprender a utilizar crop. Poderíamos muito bem utilizar a chamada abaixo, por exemplo. O que é importante é ter em mãos as coordenadas desejadas.

In [10]:
# image_crop = image_thumb.crop((0, 0, 50, 50))

## Rotação e Transposição

Outras duas operações essenciais que podemos realizar são a rotação e a transposição. Começando pela rotação, podemos utilizar o método rotate para girar a imagem. O método rotate devolve uma cópia da imagem e recebe como primeiro parâmetro o número de graus radianos que desejamos aplicar para a rotação. No exemplo abaixo, rodamos 45 graus. O método roda a imagem no sentido anti-horário e também pode receber um parâmetro fillcolor, que define a cor de preenchimento do fundo. Por padrão, o fundo é colorido de preto. Esse parâmetro pode receber nomes de cores, código RGB, hexadecimal, etc.

In [11]:
image_rotated = image_thumb.rotate(45, fillcolor="black")

image_rotated.save(FIGURES_DIR / "figura_05.jpg")

![Figura 5](assets/images/figura_05.jpg)

Percebeu que uma parte do conteúdo da imagem foi descartada? Podemos mudar esse comportamento utilizando o parâmetro expand, que definirá se a imagem de resultado deverá ser expandida para conter todo o conteúdo da imagem original.

In [12]:
image_rotated = image_thumb.rotate(45, fillcolor="#2e2e2e", expand=True)

image_rotated.save(FIGURES_DIR / "figura_06.jpg")

![Figura 6](assets/images/figura_06.jpg)

A transposição é o espelhamento da imagem. Para espelhar uma imagem, utilizamos o método transpose. Passando como argumento o modo de espelhamento, que pode ser Transpose.FLIP_LEFT_RIGHT para espelhar da esquerda para a direita ou Transpose.FLIP_TOP_BOTTOM para espelhar de baixo para cima.

In [13]:
from PIL.Image import Transpose

image_transposed = image_thumb.transpose(Transpose.FLIP_LEFT_RIGHT)

image_transposed.save(FIGURES_DIR / "figura_10.jpg")

![Figura 10](assets/images/figura_10.jpg)

## Ponte para o NumPy

Ao alimentar uma rede neural com imagens ou vídeos, nem sempre utilizaremos o PIL, que é suportado diretamente pelo PyTorch. Portanto, continuaremos na sequência aprendendo a manipular as imagens pré-processadas pelo PIL utilizando o NumPy. Para converter uma imagem do PIL para um formato nativo do NumPy, podemos chamar o método array da biblioteca numpy, passando como parâmetro a imagem do PIL. Como resultado, teremos uma matriz numérica multidimensional do NumPy que representa a mesma imagem em um formato compreendido globalmente por praticamente qualquer biblioteca de machine learning e deep learning.

In [14]:
import numpy

numeric_matrix = numpy.array(image_thumb)

print(numeric_matrix)

[[[ 12  25   7]
  [ 12  25   7]
  [ 12  25   7]
  ...
  [ 31  73  33]
  [ 49  89  50]
  [ 58  96  58]]

 [[ 12  25   7]
  [ 12  25   7]
  [ 12  25   7]
  ...
  [ 22  67  26]
  [ 37  79  39]
  [ 48  90  49]]

 [[ 12  25   7]
  [ 12  25   7]
  [ 12  25   7]
  ...
  [ 18  64  22]
  [ 24  70  26]
  [ 30  75  30]]

 ...

 [[ 81 151  30]
  [ 85 153  35]
  [ 90 154  43]
  ...
  [ 78 103  59]
  [ 71  97  55]
  [ 71  97  56]]

 [[ 81 151  28]
  [ 85 153  34]
  [ 88 153  41]
  ...
  [ 80 103  61]
  [ 74  98  57]
  [ 71  97  57]]

 [[ 81 151  29]
  [ 85 154  33]
  [ 87 153  40]
  ...
  [ 82 103  62]
  [ 75  98  58]
  [ 71  97  57]]]


Também podemos construir uma imagem do PIL com base em uma matriz numérica gerada pelo NumPy. Basta chamar o método fromarray da classe Image da biblioteca PIL, passando como primeiro parâmetro a matriz numérica.

In [15]:
image_reconstructed = Image.fromarray(numeric_matrix)